In [2]:
!pip install chromadb 

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 61.3 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 22.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 65.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 69.5 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.1/72.1 kB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 180.2/180.2 kB 15.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.0/69.0 kB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 231.6/231.6 kB 16.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 5.1 MB/s eta 0:00:00
  Attempting uninstall: opentelemetry-proto
    Found existing installation: opentelemetry-proto 1.38.0
    

In [3]:
import pandas as pd
import torch
import chromadb
from chromadb import PersistentClient
from chromadb.api.types import EmbeddingFunction, Documents, Embeddings
from transformers import AutoTokenizer, AutoModel
import math

In [4]:
CSV_DATA_PATH = "/kaggle/input/datasets/yassinmahmoud05/merged-airbnb/merged_airbnb_data.csv"
MODEL_DIR = "/kaggle/input/models/yassinmahmoud05/my-real-estate-distilbert/pytorch/default/1/my_real_estate_distilbert" 
CHROMA_DB_PATH = "/kaggle/working/"

In [5]:
COLLECTION_NAME = "seattle_airbnb_inventory"

In [6]:
class DistilBERTMeanPoolingEmbeddingFunction(EmbeddingFunction):
    def __init__(self, model_path: str):
        print(f"Loading local model and tokenizer from {model_path}...")
        self.tokenizer = AutoTokenizer.from_pretrained(model_path)
        self.model = AutoModel.from_pretrained(model_path)
        
        
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.model.to(self.device)
        self.model.eval()
        print(f"Model loaded successfully on {self.device}.")

    def __call__(self, input: Documents) -> Embeddings:
        encoded_input = self.tokenizer(
            input, 
            padding=True, 
            truncation=True, 
            return_tensors='pt',
            max_length=512,
            return_token_type_ids=False 
        ).to(self.device)

       
        with torch.no_grad():
            
            encoded_input.pop("token_type_ids", None) 
            
            model_output = self.model(**encoded_input)

        
        token_embeddings = model_output[0] 
        attention_mask = encoded_input['attention_mask']

        input_mask_expanded = attention_mask.unsqueeze(-1).expand(token_embeddings.size()).float()
        sum_embeddings = torch.sum(token_embeddings * input_mask_expanded, 1)
        sum_mask = torch.clamp(input_mask_expanded.sum(1), min=1e-9)
        mean_pooled = sum_embeddings / sum_mask

        # 4. Return as a standard Python list for ChromaDB
        return mean_pooled.cpu().numpy().tolist()

In [7]:
df = pd.read_csv(CSV_DATA_PATH)

documents = df['comments'].tolist()

ids = df['chroma_review_id'].astype(str).tolist()

metadata_cols = [col for col in df.columns if col not in ['comments', 'chroma_review_id']]

metadatas = df[metadata_cols].to_dict(orient='records')

In [8]:
chroma_client = PersistentClient(path=CHROMA_DB_PATH)


embedding_func = DistilBERTMeanPoolingEmbeddingFunction(MODEL_DIR)

collection = chroma_client.get_or_create_collection(
    name=COLLECTION_NAME,
    embedding_function=embedding_func,
    metadata={"hnsw:space": "cosine"} 
)

Loading local model and tokenizer from /kaggle/input/models/yassinmahmoud05/my-real-estate-distilbert/pytorch/default/1/my_real_estate_distilbert...


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertModel LOAD REPORT from: /kaggle/input/models/yassinmahmoud05/my-real-estate-distilbert/pytorch/default/1/my_real_estate_distilbert
Key                   | Status     |  | 
----------------------+------------+--+-
pre_classifier.weight | UNEXPECTED |  | 
classifier.weight     | UNEXPECTED |  | 
pre_classifier.bias   | UNEXPECTED |  | 
classifier.bias       | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Model loaded successfully on cuda.


In [9]:
BATCH_SIZE = 250 
total_records = len(documents)

print(f"Starting ingestion of {total_records} records into ChromaDB...")

for i in range(0, total_records, BATCH_SIZE):
    end_idx = min(i + BATCH_SIZE, total_records)
    
    batch_ids = ids[i:end_idx]
    batch_documents = documents[i:end_idx]
    batch_metadatas = metadatas[i:end_idx]
    
    # This single line handles the embedding and the saving automatically
    collection.add(
        ids=batch_ids,
        documents=batch_documents,
        metadatas=batch_metadatas
    )
    
    # Print progress so we know it hasn't frozen
    if i % 1000 == 0 or i == 0:
        print(f"Inserted records {i} to {end_idx}...")

Starting ingestion of 81491 records into ChromaDB...
Inserted records 0 to 250...
Inserted records 1000 to 1250...
Inserted records 2000 to 2250...
Inserted records 3000 to 3250...
Inserted records 4000 to 4250...
Inserted records 5000 to 5250...
Inserted records 6000 to 6250...
Inserted records 7000 to 7250...
Inserted records 8000 to 8250...
Inserted records 9000 to 9250...
Inserted records 10000 to 10250...
Inserted records 11000 to 11250...
Inserted records 12000 to 12250...
Inserted records 13000 to 13250...
Inserted records 14000 to 14250...
Inserted records 15000 to 15250...
Inserted records 16000 to 16250...
Inserted records 17000 to 17250...
Inserted records 18000 to 18250...
Inserted records 19000 to 19250...
Inserted records 20000 to 20250...
Inserted records 21000 to 21250...
Inserted records 22000 to 22250...
Inserted records 23000 to 23250...
Inserted records 24000 to 24250...
Inserted records 25000 to 25250...
Inserted records 26000 to 26250...
Inserted records 27000 to 

In [10]:
import shutil
from IPython.display import FileLink

print("Zipping up the ChromaDB memory...")
# This bundles everything in your working directory into a single zip file
shutil.make_archive("/kaggle/working/seattle_rag_memory", 'zip', "/kaggle/working")

print("Ready for download!")
# Generates the clickable link
FileLink(r'seattle_rag_memory.zip')

Zipping up the ChromaDB memory...
Ready for download!


/kaggle/working/seattle_rag_memory.zip